# 1 Imports

In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold

import lightgbm as lgbm

import funciones
from funciones import *
import joblib
import importlib

c:\Users\Jorge\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
importlib.reload(funciones)

<module 'funciones' from 'c:\\Users\\Jorge\\Documents\\Proyectos de programación\\TFG\\Modelos\\Jorge\\funciones.py'>

# 2 Carga de datos

In [4]:
SPOOF = "..\\..\\Dataset\\Features\\spoof_features.csv"
BONAFIDE = '..\\..\\Dataset\\Features\\bonafide_features.csv'

RANDOM_STATE = 12
TEST_SPLIT = 0.2

In [5]:
df = carga_datos(ruta_bon=BONAFIDE, ruta_spo=SPOOF, random_state=RANDOM_STATE)

In [6]:
X = df.drop(columns=['label', 'filename'])
y = df['label']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SPLIT, random_state=RANDOM_STATE, stratify=y)

# 3 Entrenamiento de modelos

## 3.1 Modelo baseline

In [8]:
train_data_lgbm = lgbm.Dataset(X_train, label=y_train)
test_data_lgbm = lgbm.Dataset(X_test, label=y_test, reference=train_data_lgbm)

In [9]:
lgbm_base_params = {
    'objective': 'binary',              # Clasificación binaria
    'metric': 'binary_logloss',          # Métrica de evaluación
    'boosting_type': 'gbdt',              # Gradient Boosting Decision Tree
    'num_leaves': 31,                     # Número máximo de hojas (controla complejidad)
    'learning_rate': 0.05,                 # Tasa de aprendizaje (eta)
    'feature_fraction': 0.9,               # Fracción de características por iteración
    'bagging_fraction': 0.8,               # Fracción de datos por iteración
    'bagging_freq': 5,                      # Frecuencia de bagging
    'verbose': 0,                           # Modo silencioso
    'random_state': RANDOM_STATE,
    'is_unbalance': True,                    # Compensación automática de clases desbalanceadas
    'num_threads': -1                        # Usar todos los cores
}

In [10]:
lgbm_base_model = lgbm.train(
    lgbm_base_params,
    train_data_lgbm,
    valid_sets=[test_data_lgbm],      # Evaluación en validación
    num_boost_round=1000,         # Número máximo de iteraciones
    callbacks=[
        lgbm.early_stopping(50),   # Early stopping si no mejora en 50 rondas
        lgbm.log_evaluation(100)    # Log cada 100 iteraciones
    ]
)

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

In [11]:
y_pred_proba_base_lgbm = lgbm_base_model.predict(X_test)
y_pred_base_lgbm = (y_pred_proba_base_lgbm >= 0.5).astype(int)

In [12]:
svm_base_metrics = evaluacion_modelo(
    "LightGBM - Baseline", 
    y_test, 
    y_pred_base_lgbm, 
    y_pred_proba_base_lgbm,
)


 EVALUACIÓN: LightGBM - Baseline

MÉTRICAS PRINCIPALES:
   • Accuracy:  1.0000  (Porcentaje total de aciertos)
   • Precision: 1.0000  (De los que dije que eran reales, ¿cuántos lo eran?)
   • Recall:    1.0000  (De los reales, ¿cuántos detecté?)
   • F1-Score:  1.0000  (Balance precision-recall)
   • AUC-ROC:   1.0000  (Capacidad discriminativa)
   • MCC:       1.0000  (Coeficiente de correlación de Matthews)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Real (1)     1.0000    1.0000    1.0000        10
      IA (0)     1.0000    1.0000    1.0000        10

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20


ANÁLISIS DE ERRORES:
   • Verdaderos Negativos (Reales bien clasificados): 10
   • Verdaderos Positivos (IA bien clasificados): 10
   • Falsos Positivos (Reales clasificados como IA): 0
   • Falsos Negativos (IA clasificados como Rea

## 3.2 Cross validation

In [13]:
# Para validación cruzada con LightGBM usamos la interfaz scikit-learn
lgbm_cv = lgbm.LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    num_leaves=31,
    learning_rate=0.05,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=RANDOM_STATE,
    is_unbalance=True,
    n_jobs=-1,
    verbose=-1
)

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores_accuracy = cross_val_score(lgbm_cv, X_train, y_train, cv=cv, scoring='accuracy')
cv_scores_f1 = cross_val_score(lgbm_cv, X_train, y_train, cv=cv, scoring='f1')
cv_scores_auc = cross_val_score(lgbm_cv, X_train, y_train, cv=cv, scoring='roc_auc')

In [15]:
print("\nResultados Validación Cruzada (5 folds):")
print(f"   • Accuracy: {cv_scores_accuracy.mean():.4f} (+/- {cv_scores_accuracy.std()*2:.4f})")
print(f"   • F1-Score: {cv_scores_f1.mean():.4f} (+/- {cv_scores_f1.std()*2:.4f})")
print(f"   • AUC-ROC:  {cv_scores_auc.mean():.4f} (+/- {cv_scores_auc.std()*2:.4f})")


Resultados Validación Cruzada (5 folds):
   • Accuracy: 0.9875 (+/- 0.0500)
   • F1-Score: 0.9882 (+/- 0.0471)
   • AUC-ROC:  1.0000 (+/- 0.0000)


## 3.3 Grid Search

In [16]:
param_grid = {
    'num_leaves': [15, 31, 63],           # Controla complejidad del modelo
    'learning_rate': [0.01, 0.05, 0.1],    # Tasa de aprendizaje
    'n_estimators': [100, 200, 300],       # Número de iteraciones
    'min_child_samples': [20, 50, 100],     # Mínimo muestras por hoja
    'subsample': [0.6, 0.8, 1.0],           # Fracción de muestras (bagging)
    'colsample_bytree': [0.6, 0.8, 1.0],    # Fracción de características
    'reg_alpha': [0, 0.1, 0.5],             # Regularización L1
    'reg_lambda': [0, 0.1, 0.5]             # Regularización L2
}

In [17]:
# Grid Search con validación cruzada
lgbm_classifier = lgbm.LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    random_state=RANDOM_STATE,
    is_unbalance=True,
    n_jobs=-1,
    verbose=-1
)

In [18]:
grid_search = GridSearchCV(
    lgbm_classifier,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 6561 candidates, totalling 32805 fits


,estimator,"LGBMClassifie...2, verbose=-1)"
,param_grid,"{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'min_child_samples': [20, 50, ...], 'n_estimators': [100, 200, ...], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,boosting_type,'gbdt'


In [23]:
# Modelo optimizado
lgbm_optimized = grid_search.best_estimator_

In [24]:
for param in grid_search.param_grid.keys():
    print(f"{param}: {grid_search.best_params_[param]}")

num_leaves: 15
learning_rate: 0.01
n_estimators: 100
min_child_samples: 20
subsample: 0.6
colsample_bytree: 0.6
reg_alpha: 0
reg_lambda: 0


In [25]:
y_pred_proba_optimized_lgbm = lgbm_optimized.predict_proba(X_test)[:, 1]
y_pred_optimized_lgbm = lgbm_optimized.predict(X_test)

In [26]:
optimized_metrics = evaluacion_modelo(
    "LightGBM - Optimizado", 
    y_test, 
    y_pred_optimized_lgbm, 
    y_pred_proba_optimized_lgbm,
)


 EVALUACIÓN: LightGBM - Optimizado

MÉTRICAS PRINCIPALES:
   • Accuracy:  1.0000  (Porcentaje total de aciertos)
   • Precision: 1.0000  (De los que dije que eran reales, ¿cuántos lo eran?)
   • Recall:    1.0000  (De los reales, ¿cuántos detecté?)
   • F1-Score:  1.0000  (Balance precision-recall)
   • AUC-ROC:   1.0000  (Capacidad discriminativa)
   • MCC:       1.0000  (Coeficiente de correlación de Matthews)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Real (1)     1.0000    1.0000    1.0000        10
      IA (0)     1.0000    1.0000    1.0000        10

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20


ANÁLISIS DE ERRORES:
   • Verdaderos Negativos (Reales bien clasificados): 10
   • Verdaderos Positivos (IA bien clasificados): 10
   • Falsos Positivos (Reales clasificados como IA): 0
   • Falsos Negativos (IA clasificados como R

# 4 Guardado del mejor modelo

In [27]:
# Guardar el modelo
joblib.dump(lgbm_optimized, 'modelo_LightGBM.pkl')

['modelo_LightGBM.pkl']